# Browse the hierarchy as a DataTree

The ISMIP7 forcing datasets on [source.coop](https://source.coop/ismip) are about
285,000 NetCDF files (6 TB). The [englacial/ismip7](https://source.coop/englacial/ismip7)
Icechunk stores hold *virtual references* into those files (22 GB of metadata, no data
copied). One store per ice sheet, `icechunk-ais` and `icechunk-gris`, laid out as

    <ESM>/<experiment>/<product>/<resolution>m      e.g. CESM2-WACCM/ssp585/SDBN1/2000m

Each resolution group is one xarray Dataset: a normalised monthly (or annual) time axis
in `days since 1850-01-01`, the exact ISMIP7 grid in `x`/`y`, and a `crs` variable with
the EPSG code and WKT. The group above the resolutions (the *family*, e.g.
`CESM2-WACCM/ssp585/SDBN1`) carries an OME-NGFF `multiscales` attribute naming its
levels, so a multiscale-aware reader (gridlook) opens the family as one pyramid.

Reading needs no credentials. Everything below uses the standard tools, `icechunk`,
`zarr` and `xarray`; nothing from `ismip7_helper`.


In [ ]:
import os
import time

import icechunk
import xarray as xr
import zarr

REGION = os.environ.get("ISMIP7_NB_REGION", "AIS")  # or "GrIS"
BUCKET = "us-west-2.opendata.source.coop"
PREFIX = f"englacial/ismip7/icechunk-{REGION.lower()}"
# the bucket the virtual chunks point into; icechunk only follows references into
# containers you authorise explicitly
SOURCE = "s3://us-west-2.opendata.source.coop/ismip/"


# the series run to 2300, past datetime64[ns]: decode time with cftime
TIME = xr.coders.CFDatetimeCoder(use_cftime=True)


def read_time(a, tries=3):
    """Seconds to load an xarray object into memory (retrying transient S3 errors)."""
    for attempt in range(tries):
        t0 = time.time()
        try:
            a.load()
            return time.time() - t0
        except Exception:
            if attempt == tries - 1:
                raise
            time.sleep(2)


storage = icechunk.s3_storage(bucket=BUCKET, prefix=PREFIX, region="us-west-2", anonymous=True)
containers = icechunk.containers_credentials({SOURCE: icechunk.s3_anonymous_credentials()})
repo = icechunk.Repository.open(storage, authorize_virtual_chunk_access=containers)
session = repo.readonly_session("main")
print("store:", f"s3://{BUCKET}/{PREFIX}", "| snapshot", session.snapshot_id)

In [ ]:
root = zarr.open_group(session.store, mode="r")

families, levels = {}, []
for path, node in root.members(max_depth=None):
    if not isinstance(node, zarr.Group):
        continue
    if "multiscales" in node.attrs:
        families[path] = [d["path"] for d in node.attrs["multiscales"][0]["datasets"]]
    elif node.attrs.get("ismip7_group"):
        levels.append(path)
print(len(levels), "data groups,", len(families), "families")
for fam, lv in list(families.items())[:8]:
    print(f"  {fam}: {lv}")
FAMILY = os.environ.get("ISMIP7_NB_FAMILY", "CESM2-WACCM/historical/SDBN1")
GROUP = f"{FAMILY}/{families[FAMILY][-1]}"  # the coarsest level
print("using", GROUP)

## Scope the tree

`xr.open_datatree` reads the metadata of every group below the one you open, so open the family or the ESM you need rather than the whole store.

In [ ]:
# the family is small and opens in a moment; an ESM or the whole store means one
# metadata read per group (hundreds), so scope the tree to what you need
t0 = time.time()
dt = xr.open_datatree(
    session.store, group=FAMILY, engine="zarr", consolidated=False, decode_times=TIME
)
print(f"{FAMILY}: {len(list(dt.subtree))} nodes in {time.time() - t0:.1f}s")
for node in dt.subtree:
    if node.data_vars:
        print(" ", node.path, sorted(node.data_vars)[:6], "...")

In [ ]:
ESM = FAMILY.split("/")[0]
t0 = time.time()
dt = xr.open_datatree(
    session.store, group=ESM, engine="zarr", consolidated=False, decode_times=TIME
)
leaves = [n.path for n in dt.subtree if n.data_vars]
print(f"{ESM}: {len(leaves)} data groups in {time.time() - t0:.1f}s")
print(leaves[:12])

## Optional: a faster walker

In [ ]:
# Optional: rustytree (https://github.com/aladinor/rustytree) walks a Zarr v3 hierarchy
# in Rust, concurrently, as `engine="rustytree"`; install it from its repository
# (pip install git+https://github.com/aladinor/rustytree) to compare.
try:
    import rustytree  # noqa: F401
except ImportError:
    print("rustytree is not installed; skipping")
else:
    for engine in ("zarr", "rustytree"):
        t0 = time.time()
        dt = xr.open_datatree(session.store, group=ESM, engine=engine, decode_times=TIME)
        print(f"{engine:9s}: {sum(1 for _ in dt.subtree)} nodes in {time.time() - t0:.1f}s")